# American Express Data Science Coding Practice 

### ML Ques

#### Q1. Random Forest v/s Gradient Boosting

#### Problem
Interviewers directly ask: 'Explain the difference between Random Forest and Gradient-Boosted Decision Trees.' Show a code example using sklearn to train and compare both on the same data.

---

#### Example

**Random Forest** = parallel trees (bagging)     **Gradient Boosting** = sequential trees (each corrects previous)



In [2]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [3]:
# n_samples = number of rows = 1000
# n_features = number of columns  = 10
# n_informative = 5 . Means Out of 10 columns only 5 columns are useful for prediction. Rest 5 columns are noise.
# random_state = 42 . Data stays same in every run.

X,y = make_classification(n_samples = 1000, n_features = 10 , n_informative= 5, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2)

##### Random Forest

In [ ]:
# n_estimators = 100 . Number of trees in the Random Forest.
# random_state = 42 . Data stays same in every run.

rf = RandomForestClassifier(n_estimators = 100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

accuracy_score_rf = accuracy_score(y_test, y_pred_rf) # Accuracy of Random Forest model on the test set.

print("Random Forest Accuracy:", accuracy_score_rf)


Random Forest Accuracy: 0.945


##### Gradient Boosting

In [9]:
gb = GradientBoostingClassifier(n_estimators= 100, random_state = 42)
gb.fit(X_train, y_train)
y_pred_gb = gb.predict(X_test) 

accuracy_score_gb = accuracy_score(y_test,y_pred_gb) # Accuracy of Gradient Boosting model on the test set.

print("Gradient Boosting Accuracy:" , accuracy_score_gb)



Gradient Boosting Accuracy: 0.93


#### Q2. Cross Validation To Detect Overfitting

#### Problem
Overfitting is when a model memorizes training data but fails on new data. Demonstrate using k-fold cross-validation to detect and prevent it — a key concept for model validation at Amex.

---

#### Example

**Training Accuracy** = 99%,  **Test Accuracy** = 70% → Model is overfitting. **Cross-Val Score** ≈ 72% reveals true performance.



In [11]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import cross_val_score, KFold
import numpy as np

In [12]:
X, y = make_classification(n_samples = 1000, n_features = 10, random_state = 42)


In [13]:
overfitting_decision_tree_model = DecisionTreeClassifier(max_depth= None)

In [14]:
good_decision_tree_model = DecisionTreeClassifier(max_depth= 3)


In [ ]:
# Split data into 5 folds for cross-validation, shuffle data before splitting,
# and use random_state=42 to ensure same splits every run
# KFold is used because normarla train_test_split divides datasets in percentage. Meanwhile KFold is used to divide data set in such a manner that each data point
# atlesast once get a chance to be in the test set.

kf = KFold(n_splits = 5, shuffle = True, random_state = 42)  

In [ ]:
scores_of_overfitting_decision_tree_model = cross_val_score(overfitting_decision_tree_model, X, y, cv=kf)

scores_of_good_decision_tree_model = cross_val_score(good_decision_tree_model, X, y, cv = kf)

In [17]:
overfitting_mean = scores_of_overfitting_decision_tree_model.mean()
overfitting_std = scores_of_overfitting_decision_tree_model.std()

print(f"Overfitting Decision Tree Model Scores: {overfitting_mean} +/- {overfitting_std}")

Overfitting Decision Tree Model Scores: 0.874 +/- 0.023108440016582705


In [18]:
good_mean = scores_of_good_decision_tree_model.mean()
good_std = scores_of_good_decision_tree_model.std()

print(f"Good Decision Tree Model Scores: {good_mean} +/- {good_std}")

Good Decision Tree Model Scores: 0.8720000000000001 +/- 0.009273618495495713


#### Q3. K-Means Clustering For Customer Segmentation

#### Problem
Amex case study: segment customers into groups (high-value, at-risk, growth) using K-Means clustering. This is the 'Customer Segmentation' case study confirmed in Amex interviews.

---

#### Example

**Input:** Customer Spend & Recency Data   **Output:**  3 Clusters → High Value / At Risk / Growth



In [1]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import numpy as np

In [29]:
np.random.seed(7)

# Below code creates 6 x 3 matrix.

customers = np.array([
    [5,  20, 5000],   # recent, frequent, high spend → VIP
    [100, 2,  200],    # long ago, rare, low spend → at risk
    [10, 15, 3000],   # recent, moderate → growth
    [90,  1,  100],    # very dormant
    [3,  25, 8000],   # super VIP
    [60,  5,  500],    # occasional
])

print(customers)

[[   5   20 5000]
 [ 100    2  200]
 [  10   15 3000]
 [  90    1  100]
 [   3   25 8000]
 [  60    5  500]]


In [47]:
# Preproccesing Step
scaler = StandardScaler()
X_scaled = scaler.fit_transform(customers)

In [31]:
k_means = KMeans(n_clusters= 3, random_state = 42, n_init = 10)
labels = k_means.fit_predict(X_scaled)

In [32]:
for i in range(len(customers)):

    cust = customers[i]
    label = labels[i]

    print("Customer", i+1, ":", cust, "→ Cluster", label)

Customer 1 : [   5   20 5000] → Cluster 0
Customer 2 : [100   2 200] → Cluster 1
Customer 3 : [  10   15 3000] → Cluster 0
Customer 4 : [ 90   1 100] → Cluster 1
Customer 5 : [   3   25 8000] → Cluster 2
Customer 6 : [ 60   5 500] → Cluster 1


#### Q4. Cosine Similarity Between Two Text Embeddings

#### Problem
LLMs represent text as vectors (embeddings). Cosine similarity measures how semantically similar two texts are. Essential for RAG (Retrieval-Augmented Generation) systems and LLM validation.

---

#### Example

**Credit Card Fraud** vs **payment fraud** → similarity: 0.92 (very similar) **credit card fraud** vs **yoga class** → similarity: 0.15



In [44]:
import numpy as np

In [45]:
def cosine_similarity(vec1, vec2):
    similarity_scores = np.dot(vec1, vec2) / (                             # np.dot(vec1, vec2) 👉 This is dot product.
        np.linalg.norm(vec1) * np.linalg.norm(vec2)                        # np.linalg.norm(vec1) 👉 This is length (magnitude) of vector. √(x² + y² + ...)
    )                                                                      
                                                                           # (dot product) / (length of vec1 × length of vec2)
    return similarity_scores

# linalg = Linear Algebra
# .norm means: length (size) of a vector

In [ ]:
'''
      | Axis     | Meaning                  |
| -------- | ------------------------ |
| `axis=0` | Work **down columns** ⬇️ |
| `axis=1` | Work **across rows** ➡️  |


'''

In [49]:
# Simplified demo with small vectors
emb_fraud   = [0.8, 0.5, 0.1]
emb_payment = [0.7, 0.6, 0.2]
emb_yoga    = [0.1, 0.0, 0.9]

print(f"Fraud vs Payment: {cosine_similarity(emb_fraud, emb_payment):.3f}")
print(f"Fraud vs Yoga:    {cosine_similarity(emb_fraud, emb_yoga):.3f}")

Fraud vs Payment: 0.983
Fraud vs Yoga:    0.198


### More Ques
